# clue_roll

> Phase 12: batched rollouts on the clue board with a policy (and a memory) per agent slot and scripted liars, rendering every prompt as `LLMMember` does, so that with the same replies the boards, rows and turn logs equal `runClue`'s; rows with the Phase 11 columns plus solo solves, wrong answers per solve and follow share; and `runEval` for evaluation cells.

In [ ]:
#| default_exp clue_roll

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace

import numpy as np

from explRL.bench_grid import readCSV, rowKey, writeCSV
from explRL.clue_agents import ClueLiar
from explRL.clue_fix import fixOptions
from explRL.clue_policy import renderPrompt
from explRL.clue_runs import NAN, clueMetrics
from explRL.clue_train import ClueEpisodes
from explRL.llm_agent import _plain
from explRL.llm_runs import LLMConfig, cellRow, llmStats
from explRL.local_policy import LocalPolicy
from explRL.rl_torch import freeGPU

SUGGEST = re.compile(r'The coordinator suggests: T(\d+)\.')


@dataclass
class Rollout:
    cfg: object
    seeds: list
    boards: list
    logs: list          # logs[b][slot]: LLMMember-format turn records of the text slots
    returns: object     # (boards x N): delivered rewards of text slots plus rewards pending at the end
    prompts: list = None   # prompts[b][slot]: (system, blocks) per turn, when kept (training)


def _per(x, i):
    return x[i] if isinstance(x, dict) else x


def rollout(cfg, seeds, policies, scripted=None, memory='notes', keepPrompts=False):
    """Play a batch of clue boards to the end: per round, every active board's pending text turn is rendered (as
    LLMMember would) and each policy answers its requests in one call; scripted slots act on the board directly."""
    w, B, N = cfg.world, len(seeds), cfg.N
    ep = ClueEpisodes(B, N, w, w.get('scheme', 'individual'), cfg.turnsPerAgent(), channel=cfg.channel, seeds=seeds,
                      scripted=scripted, text=dict(hint=w.get('hint'), showOpen=w.get('showOpen', False), **fixOptions(w)),
                      passPooled=w.get('passPooled', False))
    notes, past = [[''] * N for _ in range(B)], [[[] for _ in range(N)] for _ in range(B)]
    logs, R = [[[] for _ in range(N)] for _ in range(B)], np.zeros((B, N))
    prompts, asked = [[[] for _ in range(N)] for _ in range(B)], [None] * B
    nxt = ep.reset()
    while any(x is not None for x in nxt):
        groups = {}
        for b, x in enumerate(nxt):
            if x is not None:
                i, obs = x
                pol = _per(policies, i)
                prompt = renderPrompt(ep.envs[b], obs, _per(memory, i), notes[b][i], past[b][i], cfg.framing)
                groups.setdefault(id(pol), (pol, []))[1].append((b, prompt))
                asked[b] = prompt
        replies = [None] * B
        for pol, items in groups.values():
            for (b, _), rep in zip(items, pol([p for _, p in items])):
                replies[b] = rep
        nxt, rewards, _ = ep.step([r.parsed if r is not None else None for r in replies])
        for b, rep in enumerate(replies):
            if rep is None:
                continue
            L = ep.last[b]
            i = L['agent']
            R[b, i] += L['reward']
            if isinstance(rep.parsed, dict) and isinstance(rep.parsed.get('notes'), str):
                notes[b][i] = rep.parsed['notes'][:1000]
            past[b][i].append(L['summary'])
            if keepPrompts:
                prompts[b][i].append(asked[b])
            logs[b][i].append(dict(kind='turn', agent=i, t=_plain(L['t']), obs=L['obs'], response=rep.text,
                                   action=_plain(L['action']), valid=L['valid'], reward=_plain(L['reward']),
                                   report=_plain(L['report']), explore=L['explore'], stop=rep.stopReason,
                                   refusal=rep.refusal, usage=rep.usage, cost=rep.cost))
    R += np.array([bd.pending for bd in ep.boards])
    return Rollout(cfg, list(seeds), ep.boards, logs, R, prompts if keepPrompts else None)


def coopMetrics(board, logs):
    "Solo solves (first correct answers while more than one candidate was left), wrong answers per solve, follow share."
    solved, wrong = board.counts['solved'], board.counts['wrong']
    solo = sum(1 for s in board.submits if s[4] == 'solved' and s[3] is not None and s[3] > 1)
    opens = [(int(m.group(1)), x['action'][1]) for lg in logs for x in lg
             if x['action'][0] == 'open' and (m := SUGGEST.search(x['obs']))]
    return dict(soloSolves=solo, soloShare=solo / solved if solved else NAN, wrongPerSolve=wrong / solved if solved else NAN,
                followShare=float(np.mean([a == b for a, b in opens])) if opens else NAN)


def coopRows(cfg, ro, honest=None):
    "One row per board: the Phase 11 columns (clueMetrics, llmStats over the text slots) plus coopMetrics."
    honest = range(cfg.nLiar, cfg.N) if honest is None else honest
    out = []
    for b, bd in enumerate(ro.boards):
        members = [SimpleNamespace(log=lg) for lg in ro.logs[b] if lg]
        out.append(dict(episode=0, **clueMetrics(cfg, bd, None, honest, cfg.turnsPerAgent()), **llmStats(members),
                        **coopMetrics(bd, ro.logs[b])))
    return out


def runEval(cells, policyFor, logDir=None):
    """Run evaluation cells (llmCell dicts, task 'clue') with `rollout`, one batch per cell configuration; liars
    (cfg.nLiar) in the first slots. policyFor(cell) -> (policies, memory). Rows keyed with cellRow, sorted by key."""
    groups = {}
    for c in cells:
        groups.setdefault(json.dumps(dict(c, seed=None), sort_keys=True, default=str), []).append(c)
    rows = []
    for cs in groups.values():
        cfg = LLMConfig.fromDict(cs[0]['cfg'])
        policies, memory = policyFor(cs[0])
        ro = rollout(cfg, [c['seed'] for c in cs], policies, {i: ClueLiar for i in range(cfg.nLiar)}, memory)
        for b, (c, r) in enumerate(zip(cs, coopRows(cfg, ro))):
            row = cellRow(c, cfg, r)
            rows.append(row)
            if logDir:
                p = Path(logDir) / (rowKey(row).replace('|', '__') + '.jsonl')
                p.parent.mkdir(parents=True, exist_ok=True)
                recs = sorted((x for lg in ro.logs[b] for x in lg), key=lambda x: (x['t'], x['agent']))
                p.write_text(''.join(json.dumps(dict(key=rowKey(row), episode=0, **x), sort_keys=True) + '\n' for x in recs))
    return sorted(rows, key=rowKey)


def evalModels(cells, models, out, cache=None, logDir=None):
    """Evaluate local models, one at a time: models = {name: (model path, adapter path or None)}, cells levelled
    '<name>|<test>'. Each model's rows go to out/<name>.csv as soon as it is done, and a model already done is read back,
    not rerun (a rerun after a disconnect continues). Models on the same weights share one engine (the RL adapters on
    the warm-start model), loaded once; an engine is freed before the next is loaded."""
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    rows, engine, pol = [], None, None
    for name, (path, adapter) in models.items():
        mine = [c for c in cells if c['level'].startswith(name + '|')]
        if (out / f'{name}.csv').exists():
            rows += readCSV(out / f'{name}.csv')
            continue
        if not mine:
            continue
        if engine is None or engine.model != path:
            freeGPU(*[p for p in (pol, engine) if p is not None])
            engine = LocalPolicy(path, cache=cache)
        pol = engine if adapter is None else LocalPolicy(path, adapter=adapter, cache=cache, share=engine, tokenizer=engine.tok())
        t0 = time.time()
        got = runEval(mine, lambda c: (pol, 'notes'), logDir)
        writeCSV(got, out / f'{name}.csv')
        rows += got
        print(f'{name}: {len(got)} boards, {sum(int(r["calls"]) for r in got)} turns, {time.time() - t0:.0f} s', flush=True)
    freeGPU(*[p for p in (pol, engine) if p is not None])
    return sorted(rows, key=rowKey)

## Tests

In [ ]:
import json
import numpy as np
from explRL.clue_roll import *
from explRL.clue_policy import claudePolicy, scriptedPolicy
from explRL.clue_runs import runClue
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
from explRL.llm_grid import llmCell
from explRL.llm_runs import LLMConfig

W = dict(m=2, r=1, R=30)
fake = lambda: LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))
# AC-3 fidelity: with the same replies, rollout rows equal runClue rows (every channel, liars, both memories, options)
for ch, nLiar, mem, opt in (('blackboard', 0, 'full', {}), ('messages', 1, 'full', {}), ('hierarchical', 0, 'notes', {}),
                            ('none', 0, 'full', {}), ('blackboard', 1, 'notes', {}), ('hierarchical', 0, 'full', dict(passPooled=True, hint='follow', showOpen=True))):
    cfg = LLMConfig(task='clue', channel=ch, N=4, turns=8, nLiar=nLiar, memory=mem, world={**W, **opt})
    ref, refLog = runClue(cfg, 900, fake())
    ro = rollout(cfg, [900], claudePolicy(fake()), {i: ClueLiar for i in range(nLiar)}, mem)
    row = coopRows(cfg, ro)[0]
    assert {k: v for k, v in row.items() if k in ref[0]} == ref[0], (ch, nLiar, mem)
    mine = sorted((x for lg in ro.logs[0] for x in lg), key=lambda x: (x['t'], x['agent']))
    theirs = sorted(refLog, key=lambda x: (x['t'], x['agent']))
    assert [dict(x, episode=0) for x in mine] == theirs, ch
# kept prompts line up with the turn logs and are exactly what the policy was asked
asked = []
def recording(reqs):
    asked.extend(reqs)
    return claudePolicy(fake())(reqs)
cfg = LLMConfig(task='clue', channel='blackboard', N=4, turns=3, world=W)
ro = rollout(cfg, [906], recording, keepPrompts=True)
assert [len(p) for p in ro.prompts[0]] == [len(l) for l in ro.logs[0]] == [3] * 4
assert sorted(map(str, (p for ps in ro.prompts[0] for p in ps))) == sorted(map(str, asked))
assert all(p[1][-1]['text'] == 'Now:\n' + x['obs'] for ps, ls in zip(ro.prompts[0], ro.logs[0]) for p, x in zip(ps, ls))
assert rollout(cfg, [906], claudePolicy(fake())).prompts is None
# repeated seeds give identical boards; batching calls each policy once per round
calls = []
def counted(reqs):
    calls.append(len(reqs))
    return claudePolicy(fake())(reqs)
cfg = LLMConfig(task='clue', channel='blackboard', N=4, turns=5, world=W)
ro = rollout(cfg, [901, 901, 902], counted)
r = coopRows(cfg, ro)
assert r[0] == r[1] and r[0] != r[2] and len(calls) == 4 * 5 and all(c == 3 for c in calls)
# per-slot policies: slot 0 scripted (always wait), the others the fake Claude
waiter = scriptedPolicy(lambda s, b: json.dumps(dict(action='wait', target='', answer=0, clue='', notes='w')))
ro = rollout(cfg, [903], {0: waiter, 1: claudePolicy(fake()), 2: claudePolicy(fake()), 3: claudePolicy(fake())}, memory={0: 'notes', 1: 'full', 2: 'full', 3: 'full'})
assert all(x['action'][0] == 'wait' for x in ro.logs[0][0]) and len(ro.logs[0][1]) == 5
# returns include rewards still pending at the end (credit scheme): delivered + pending = the team's net value
cfg = LLMConfig(task='clue', channel='blackboard', N=4, turns=30, world={**W, 'scheme': 'credit'})
ro = rollout(cfg, [904, 905], claudePolicy(fake()))
for b, bd in enumerate(ro.boards):
    assert abs(ro.returns[b].sum() - bd.netByAgent().sum()) < 1e-9
# coopMetrics on hand-made logs: solo solves, wrong per solve, follow share
bd = SimpleNamespace(counts=dict(solved=2, wrong=3), submits=[(0, 0, 1, 4, 'wrong'), (1, 0, 1, 3, 'solved'), (2, 1, 2, 1, 'solved')])
lg = [[dict(action=['open', 3], obs='... The coordinator suggests: T3. ...'), dict(action=['open', 5], obs='The coordinator suggests: T4.')],
      [dict(action=['open', 7], obs='no suggestion'), dict(action=['wait', None], obs='The coordinator suggests: T1.')]]
assert coopMetrics(bd, lg) == dict(soloSolves=1, soloShare=0.5, wrongPerSolve=1.5, followShare=0.5)
# runEval: one rollout per configuration, rows keyed as cells, liars from nLiar
cells = [llmCell('coopEval', 'coopEval', 'm|main', 4, s, task='clue', channel='blackboard', turns=4, world=W) for s in (900, 901)] + \
        [llmCell('coopEval', 'coopEval', 'm|liar', 4, 900, task='clue', channel='messages', turns=4, nLiar=1, world=W)]
rows = runEval(cells, lambda c: (claudePolicy(fake()), 'full'))
assert [(r['level'], r['seed']) for r in rows] == [('m|liar', 900), ('m|main', 900), ('m|main', 901)] and rows[0]['calls'] == 12
assert {k: v for k, v in rows[1].items() if k in ('welfare', 'calls')} == {k: v for k, v in coopRows(LLMConfig.fromDict(cells[0]['cfg']), rollout(LLMConfig.fromDict(cells[0]['cfg']), [900], claudePolicy(fake()), memory='full'))[0].items() if k in ('welfare', 'calls')}

# Phase 13 options (AC-2): rollout rows and turn logs still equal runClue's with each option on; asks and checks are in the logs
import hashlib, re
from explRL.clue_text import TextClueEnv
from explRL.clueboard import ClueBoardEnv
def asker(req):
    "fakeReply, but some turns ask for the open subtask's clues or check the first clue on the board."
    o = req['messages'][0]['content'][-1]['text']
    h, op = int(hashlib.sha256(o.encode()).hexdigest()[:8], 16) % 4, re.search(r'Your open subtask: T(\d+), open', o)
    seen = re.search(r'(?:the board|the team): T(\d+): (x mod \d+ = -?\d+)', o)
    if h == 1 and op:
        return json.dumps(dict(action='ask', target=f'T{op.group(1)}', answer=0, clue='', notes='a'))
    if h == 2 and seen:
        return json.dumps(dict(action='check', target=f'T{seen.group(1)}', answer=0, clue=seen.group(2), notes='c'))
    return fakeReply(req)
for opt in (dict(ask=True), dict(table=True), dict(record=True), dict(check=True), dict(ask=True, table=True, record=True, check=True),
            dict(record='private'), dict(check='public'), dict(record='private', check='public')):    # + Phase 14 settings
    for nLiar, mem in ((0, 'full'), (1, 'notes')):
        cfg = LLMConfig(task='clue', channel='blackboard', N=4, turns=12, nLiar=nLiar, memory=mem, world={**W, **opt})
        ref, refLog = runClue(cfg, 900, LLMClient('claude-haiku-5-5', client=FakeClient(asker)))
        client = LLMClient('claude-haiku-5-5', client=FakeClient(asker))
        schema = TextClueEnv(ClueBoardEnv(900, 4, **W), **opt).SCHEMA          # as claudePolicy(..., schema=env.SCHEMA)
        pol = lambda reqs: [client.complete(s, b, schema, task='clue') for s, b in reqs]
        ro = rollout(cfg, [900], pol, {i: ClueLiar for i in range(nLiar)}, mem)
        assert {k: v for k, v in coopRows(cfg, ro)[0].items() if k in ref[0]} == ref[0], opt
        mine = sorted((x for lg in ro.logs[0] for x in lg), key=lambda x: (x['t'], x['agent']))
        assert [dict(x, episode=0) for x in mine] == sorted(refLog, key=lambda x: (x['t'], x['agent'])), opt
        if opt.get('ask'):
            assert any(x['report'] is not None and x['report'][0] == 'ask' and x['valid'] and '"ask"' in x['response'] for x in refLog)
        if opt.get('check'):
            assert any('Your check: T' in x['obs'] for x in refLog) and any('"check"' in x['response'] and x['valid'] for x in refLog)
        assert any('Checked by agent' in x['obs'] for x in refLog) == (opt.get('check') == 'public'), opt
        assert ('Known to the team' in refLog[-1]['obs']) == bool(opt.get('table')), opt
        assert any(', record ' in x['obs'] for x in refLog) == bool(opt.get('record') and not opt.get('table')), opt

In [ ]:
#| train
# With the [train] extra, no download: evalModels on the tiny model, one model with an adapter, and the resume
import tempfile
from peft import get_peft_model
from explRL.clue_sft import loadModel, loraConfig
from explRL.tiny_model import tinyModel
from explRL.llm_grid import llmCell
with tempfile.TemporaryDirectory() as d:
    tiny = tinyModel(d + '/tiny')
    tok, base = loadModel(tiny, 'cpu')
    get_peft_model(base, loraConfig(4, 8)).save_pretrained(d + '/adapter')       # a fresh (zero-change) adapter
    cells = [llmCell('coopEval', 'coopEval', f'{m}|main', 2, s, 'local', task='clue', channel='blackboard', turns=2,
                     memory='notes', world=W) for m in ('warm', 'credit') for s in (900, 901)]
    models = dict(warm=(tiny, None), credit=(tiny, d + '/adapter'), team=(tiny, d + '/missing'))   # team: no cells, skipped
    rows = evalModels(cells, models, d + '/eval', cache=d + '/cache', logDir=d + '/turns')
    assert sorted(r['level'] for r in rows) == ['credit|main'] * 2 + ['warm|main'] * 2 and all(int(r['calls']) == 4 for r in rows)
    assert sorted(p.name for p in Path(d + '/eval').iterdir()) == ['credit.csv', 'warm.csv']
    again = evalModels(cells, models, d + '/eval', cache=d + '/cache')             # read back, nothing rerun
    assert [rowKey(r) for r in again] == [rowKey(r) for r in rows] and [float(r['welfare']) for r in again] == [float(r['welfare']) for r in rows]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()